In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import sys
import re
import unicodedata
from collections import Counter
from pathlib import Path


In [ ]:
ex_utter = pd.read_excel("/home/vashist/Desktop/Desktop/Anita Zucker Center/Science talk retrieval/data/2026-04-17_science_talk_examples.xlsx", sheet_name="Example utterances")

ex_utter.head()



In [ ]:
seed_words = pd.read_excel("/home/vashist/Desktop/Desktop/Anita Zucker Center/Science talk retrieval/data/2026-04-17_science_talk_examples.xlsx", sheet_name="Seed words")

seed_words.head()

In [ ]:
cat_def = pd.read_excel("/home/vashist/Desktop/Desktop/Anita Zucker Center/Science talk retrieval/data/2026-04-17_science_talk_examples.xlsx", sheet_name="Category definitions")

cat_def.head()

In [ ]:
def process_utterance(df):
    df["was_sci_coded"] = (
        df["Notes"].str.contains("SCI-Coded", case=False, na=False).astype(int)
    )
    return df

ex_utter = process_utterance(ex_utter)
ex_utter.head()


In [ ]:
print(f"Raw row count: {len(ex_utter)}")
print(f"\nRaw label distribution:")
for k, v in Counter(ex_utter["Label"].fillna("__NULL__")).most_common():
    print(f"  {k}: {v}")
print(f"\nRaw setting distribution:")
for k, v in Counter(ex_utter["Setting"].fillna("__NULL__")).most_common():
    print(f"  {k}: {v}")
print(f"\nRaw source distribution:")
for k, v in Counter(ex_utter["Source"].fillna("__NULL__")).most_common():
    print(f"  {k}: {v}")

In [ ]:
def normalize_text(s):
    if pd.isna(s):
        return s
    s = unicodedata.normalize("NFKC", str(s))
    s = re.sub(r"\s+", " ", s).strip()
    return s

ex_utter["Utterance"] = ex_utter["Utterance"].apply(normalize_text)
print("Utterance text normalized (NFKC unicode + collapsed whitespace).")
ex_utter["Utterance"].head()

In [ ]:
before = len(ex_utter)
ex_utter = ex_utter.drop_duplicates(subset=["Utterance"]).reset_index(drop=True)
after = len(ex_utter)
dropped = before - after
print(f"De-duplication: dropped {dropped} duplicate utterance(s) ({before} -> {after}).")
if dropped > 0:
    print("(de-dup count is logged, not silently masked)")

In [ ]:
def parse_cue_list(s):
    if pd.isna(s) or str(s).strip() == "":
        return []
    parts = re.split(r"[|,;]", str(s))
    return [p.strip() for p in parts if p.strip()]

ex_utter["Tier2 cues"] = ex_utter["Tier2 cues"].apply(parse_cue_list)
ex_utter["Tier3 cues"] = ex_utter["Tier3 cues"].apply(parse_cue_list)

print("Tier2/Tier3 cues parsed to Python lists.")
print(f"Sample Tier2 cues (rows 5-9): {ex_utter['Tier2 cues'].iloc[5:10].tolist()}")
print(f"Sample Tier3 cues (rows 0-4): {ex_utter['Tier3 cues'].iloc[0:5].tolist()}")

In [ ]:
SETTING_VOCAB = {
    "Large Group", "Small Group", "Centers", "Transition",
    "Conversation", "Blocks/Engineering", "Unknown",
}

def normalize_setting(s):
    """Strip topic suffix (' / Foo'), map nulls to 'Unknown', snap to closed vocab."""
    if pd.isna(s) or str(s).strip() == "":
        return ("Unknown", None)
    parts = [p.strip() for p in str(s).split(" / ")]
    setting = parts[0]
    topic = " / ".join(parts[1:]) if len(parts) > 1 else None
    if setting not in SETTING_VOCAB:
        setting = "Unknown"
    return (setting, topic)

normalized = ex_utter["Setting"].apply(normalize_setting)
ex_utter["Setting"] = normalized.apply(lambda x: x[0])
ex_utter["topic"] = normalized.apply(lambda x: x[1])

print("Normalized setting distribution:")
for k, v in Counter(ex_utter["Setting"]).most_common():
    print(f"  {k}: {v}")
print(f"\nTopic non-null count: {ex_utter['topic'].notna().sum()}")
print(f"Topic values: {sorted(ex_utter['topic'].dropna().unique().tolist())}")

In [ ]:
TRANSCRIPT_REF_RE = re.compile(r"\(([^,)]+#\d+)")

def extract_transcript_ref(notes):
    if pd.isna(notes):
        return None
    m = TRANSCRIPT_REF_RE.search(str(notes))
    return m.group(1).strip() if m else None

ex_utter["transcript_ref"] = ex_utter["Notes"].apply(extract_transcript_ref)
print(f"Transcripts referenced: {ex_utter['transcript_ref'].notna().sum()} / {len(ex_utter)}")
print(f"Sample refs: {ex_utter['transcript_ref'].dropna().head(5).tolist()}")

In [ ]:
ex_utter["utt_id"] = [f"utt_{i:04d}" for i in range(len(ex_utter))]

corpus = ex_utter.rename(columns={
    "Utterance": "utterance",
    "Label": "label",
    "Setting": "setting",
    "Source": "source",
    "Tier2 cues": "tier2_cues",
    "Tier3 cues": "tier3_cues",
    "Article citation": "citation",
})[[
    "utt_id",
    "utterance",
    "label",
    "setting",
    "source",
    "tier2_cues",
    "tier3_cues",
    "was_sci_coded",
    "transcript_ref",
    "topic",
    "citation",
]]
corpus.head()

In [ ]:
print(f"Final row count: {len(corpus)}")

label_counts = Counter(corpus["label"].fillna("__NULL__"))
print(f"\nFinal label distribution:")
for k, v in label_counts.most_common():
    print(f"  {k}: {v}")

assert len(corpus) >= 195, f"Row count too low after dedup: {len(corpus)}"
assert corpus["utt_id"].is_unique, "utt_id must be unique"
assert corpus["utterance"].notna().all(), "Every row must have an utterance"
assert label_counts.get("SCIENCE_TALK", 0) >= 188, \
    f"SCIENCE_TALK count unexpectedly low: {label_counts}"
assert label_counts.get("NOT_SCIENCE_TALK", 0) >= 5, \
    f"NOT_SCIENCE_TALK count unexpectedly low: {label_counts}"

non_null = {k: v for k, v in label_counts.items() if k != "__NULL__"}
total = sum(non_null.values())
print(f"\nClass imbalance (logged, NOT silently fixed):")
for label, count in sorted(non_null.items(), key=lambda x: -x[1]):
    print(f"  {label}: {count} ({count/total:.1%})")
print(f"  null/unlabeled: {label_counts.get('__NULL__', 0)}")
print(f"\nimbalance ratio (majority/minority): {max(non_null.values()) / max(min(non_null.values()), 1):.1f}x")
print("→ Step 3 (negative mining) is the gating fix for this imbalance.")

In [ ]:
seed_words_clean = seed_words.rename(columns={
    "Term": "term",
    "Tier": "tier",
    "Category": "category",
    "Variants (optional)": "variants",
    "Curriculum_source": "curriculum_source",
    "Source_URL": "source_url",
})
seed_words_clean["variants"] = seed_words_clean["variants"].apply(parse_cue_list)
seed_words_clean["term"] = seed_words_clean["term"].apply(normalize_text)

print(f"Seed words: {len(seed_words_clean)} rows")
print(f"Tier distribution: {Counter(seed_words_clean['tier'])}")
print(f"Sample variants: {seed_words_clean[['term', 'variants']].head(5).to_dict('records')}")

In [ ]:
cat_def_clean = cat_def.rename(columns={
    "Label": "label",
    "Type": "type",
    "Definition (operational)": "definition",
    "Include if…": "include_if",
    "Exclude if…": "exclude_if",
    "Examples (teacher wording)": "examples",
})
print(f"Category definitions: {len(cat_def_clean)} rows")
cat_def_clean.head()

In [ ]:
PROCESSED_DIR = Path(
    "/home/vashist/Desktop/Desktop/Anita Zucker Center/Science talk retrieval/data/processed"
)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

corpus.to_parquet(PROCESSED_DIR / "corpus.parquet", index=False)
seed_words_clean.to_parquet(PROCESSED_DIR / "seed_words.parquet", index=False)
cat_def_clean.to_parquet(PROCESSED_DIR / "category_defs.parquet", index=False)

print(f"Wrote 3 parquets to {PROCESSED_DIR}")
for f in sorted(PROCESSED_DIR.glob("*.parquet")):
    print(f"  {f.name}: {f.stat().st_size:,} bytes")

In [ ]:
def test_corpus_schema():
    """DoD 3: load the parquet back and assert the schema."""
    df = pd.read_parquet(PROCESSED_DIR / "corpus.parquet")

    required = {
        "utt_id", "utterance", "label", "setting", "source",
        "tier2_cues", "tier3_cues", "was_sci_coded",
    }
    missing = required - set(df.columns)
    assert not missing, f"Missing required columns: {missing}"

    assert df["utt_id"].notna().all() and df["utt_id"].is_unique, \
        "utt_id must be non-null and unique"
    assert df["utterance"].notna().all(), "utterance must be non-null"

    bad_labels = set(df["label"].dropna()) - {"SCIENCE_TALK", "NOT_SCIENCE_TALK"}
    assert not bad_labels, f"Unexpected label values: {bad_labels}"

    valid_settings = {
        "Large Group", "Small Group", "Centers", "Transition",
        "Conversation", "Blocks/Engineering", "Unknown",
    }
    bad_settings = set(df["setting"].dropna()) - valid_settings
    assert not bad_settings, f"Unexpected setting values: {bad_settings}"

    sample_t2 = df["tier2_cues"].iloc[0]
    sample_t3 = df["tier3_cues"].iloc[0]
    assert isinstance(sample_t2, (list, np.ndarray)), \
        f"tier2_cues should be list-typed, got {type(sample_t2)}"
    assert isinstance(sample_t3, (list, np.ndarray)), \
        f"tier3_cues should be list-typed, got {type(sample_t3)}"

    assert df["was_sci_coded"].dtype.kind in ("i", "u", "b"), \
        f"was_sci_coded should be int/bool, got {df['was_sci_coded'].dtype}"

    seed_df = pd.read_parquet(PROCESSED_DIR / "seed_words.parquet")
    assert {"term", "tier", "category", "variants"}.issubset(seed_df.columns), \
        f"seed_words missing columns; has {list(seed_df.columns)}"

    cat_df = pd.read_parquet(PROCESSED_DIR / "category_defs.parquet")
    assert {"label", "type", "definition"}.issubset(cat_df.columns), \
        f"category_defs missing columns; has {list(cat_df.columns)}"

    print(f"Schema test PASSED")
    print(f"  corpus.parquet:        {len(df):3d} rows, {len(df.columns)} cols")
    print(f"  seed_words.parquet:    {len(seed_df):3d} rows, {len(seed_df.columns)} cols")
    print(f"  category_defs.parquet: {len(cat_df):3d} rows, {len(cat_df.columns)} cols")
    print(f"\n  corpus columns: {list(df.columns)}")

test_corpus_schema()

# Step 2 — Sub-type ("soft practice") labeling

Assign each utterance one or more practice labels:
`observation`, `prediction`, `causal_reasoning`, `evidence`, `content`.

Pipeline:
1. **Rule-based** mapping from explicit `tier2_cues` / `tier3_cues` via the Category definitions taxonomy.
2. **Keyword scan** of the utterance text against `seed_words` variants (catches cued-but-unmarked rows).
3. **LLM fallback** (currently STUB — to be replaced by the Step 0 UF LLM client).

Every row will end up with `≥1` subtype plus a `subtype_source ∈ {rule, keyword, llm}`, a `subtype_confidence ∈ [0, 1]`, and a `subtype_prompt_version` (for LLM-assigned rows only) — so any LLM assignment is auditable later.

In [ ]:
# --- OPTIONAL: swap the stub LLM for real Llama 3.3-70b ---------------------
# Step 2's three-stage classifier (rule -> keyword scan -> LLM fallback) only
# hits the LLM when an utterance has NO Tier2/Tier3 cues. On the Y1 corpus
# every row has cues, so flipping this flag fires zero API calls today; it
# matters for cue-less corpora (e.g. raw Y2 transcripts).
#
# Set USE_REAL_LLM_STEP2=True to use the real classifier. cached_request
# memoizes everything keyed on (model, prompt_version, params), so re-runs
# are free.

USE_REAL_LLM_STEP2 = False  # flip to True for cue-less corpora

from src.subtypes_2 import llm_subtype_stub, make_real_llm_subtype_classifier

if USE_REAL_LLM_STEP2:
    subtype_classifier = make_real_llm_subtype_classifier()
    print(f"Step 2 LLM fallback: REAL Llama 3.3-70b (prompt_version=subtype_v1_llama70b)")
else:
    subtype_classifier = llm_subtype_stub
    print(f"Step 2 LLM fallback: deterministic stub (prompt_version=subtype_v0_stub)")

In [ ]:
df_corpus = pd.read_parquet(PROCESSED_DIR / "corpus.parquet")
df_seed = pd.read_parquet(PROCESSED_DIR / "seed_words.parquet")
df_cat = pd.read_parquet(PROCESSED_DIR / "category_defs.parquet")

SUBTYPES_ALL = ["observation", "prediction", "causal_reasoning", "evidence", "content"]

CATEGORY_TO_SUBTYPES = {
    "INQUIRY_VERB": ["observation"],
    "COMMUNICATE_INFO": ["observation"],
    "ORGANIZE_THINKING": ["observation"],
    "ASK_QUESTION_FRAME": ["prediction"],
    "MEASUREMENT_FRAME": ["evidence"],
    "CAUSE_EFFECT_FRAME": ["causal_reasoning"],
    "REASONING_FRAME": ["causal_reasoning", "evidence"],
    "LIFE_SCIENCE_PLANTS": ["content"],
    "PHYSICAL_SCIENCE_MECHANISMS": ["content"],
    "PROPERTIES_MATERIALS": ["content"],
    "STRUCTURE_SHAPE_SPACE": ["content"],
    "MEASUREMENT_SHAPE": ["content"],
    "HYPOTHESIS_NOTE": ["prediction"],
    "FROGSTREET_CONTENT_BUCKET": [],
    "FROGSTREET_VOCAB_ROUTINE": [],
}

TERM_OVERRIDES = {
    "predict": ["prediction"],
    "prediction": ["prediction"],
    "guess": ["prediction"],
    "what if": ["prediction"],
    "hypothesis": ["prediction"],
    "hypothesise": ["prediction"],
    "hypothesize": ["prediction"],
}

seed_index = {}
for _, row in df_seed.iterrows():
    seed_index[row["term"].lower()] = row
    for v in row["variants"]:
        seed_index[v.lower()] = row

print(f"Seed lookup index built: {len(seed_index)} surface forms across {len(df_seed)} terms")
print(f"Category mapping covers: {sorted(CATEGORY_TO_SUBTYPES.keys())}")

In [ ]:
def map_cues_to_subtypes(tier2_cues, tier3_cues):
    subtypes = set()
    for cue in tier2_cues:
        cue_lc = str(cue).lower().strip()
        if cue_lc in TERM_OVERRIDES:
            subtypes.update(TERM_OVERRIDES[cue_lc])
            continue
        row = seed_index.get(cue_lc)
        if row is not None:
            subtypes.update(CATEGORY_TO_SUBTYPES.get(row["category"], []))
        else:
            subtypes.add("observation")
    if len(tier3_cues) > 0:
        subtypes.add("content")
    return sorted(subtypes)

df_corpus["subtype_rule"] = df_corpus.apply(
    lambda r: map_cues_to_subtypes(r["tier2_cues"], r["tier3_cues"]),
    axis=1,
)

n_rule = (df_corpus["subtype_rule"].apply(len) > 0).sum()
print(f"Rule-based subtype assignments: {n_rule} / {len(df_corpus)} rows")
print(f"\nSample assignments (first 5 rule-labeled rows):")
sample = df_corpus[df_corpus["subtype_rule"].apply(len) > 0].head(5)
for _, r in sample.iterrows():
    cues = list(r["tier2_cues"]) + list(r["tier3_cues"])
    print(f"  cues={cues} -> {r['subtype_rule']}")

In [ ]:
ALL_SEED_TOKENS = sorted(seed_index.keys(), key=len, reverse=True)

def scan_text_for_subtypes(text):
    """Find subtypes implied by seed tokens/variants in the utterance text."""
    if pd.isna(text):
        return []
    text_lc = text.lower()
    subtypes = set()
    for token in ALL_SEED_TOKENS:
        if not re.search(r"\b" + re.escape(token) + r"\b", text_lc):
            continue
        row = seed_index[token]
        if str(row["tier"]) == "TIER3":
            subtypes.add("content")
        elif token in TERM_OVERRIDES:
            subtypes.update(TERM_OVERRIDES[token])
        else:
            subtypes.update(CATEGORY_TO_SUBTYPES.get(row["category"], []))
    return sorted(subtypes)

df_corpus["subtype_keyword"] = df_corpus["utterance"].apply(scan_text_for_subtypes)

def kw_adds_to_rule(rule_st, kw_st):
    return sorted(set(kw_st) - set(rule_st))

df_corpus["subtype_keyword_new"] = df_corpus.apply(
    lambda r: kw_adds_to_rule(r["subtype_rule"], r["subtype_keyword"]),
    axis=1,
)

n_kw_any = (df_corpus["subtype_keyword"].apply(len) > 0).sum()
n_kw_new = (df_corpus["subtype_keyword_new"].apply(len) > 0).sum()
print(f"Keyword scan ran on all {len(df_corpus)} rows.")
print(f"  Found any seed token in:                     {n_kw_any} rows")
print(f"  Added subtype(s) not already in rule output: {n_kw_new} rows")

print(f"\nSample additive enrichments (rule + keyword adding extra subtypes):")
sample = df_corpus[df_corpus["subtype_keyword_new"].apply(len) > 0].head(5)
for _, r in sample.iterrows():
    snippet = r["utterance"][:75] + ("..." if len(r["utterance"]) > 75 else "")
    print(f"  rule={r['subtype_rule']} + keyword adds={r['subtype_keyword_new']}")
    print(f"    '{snippet}'")

In [ ]:
# Three-stage assignment using `subtype_classifier` selected in the cell above.
# When USE_REAL_LLM_STEP2=False this is the deterministic stub; when True it's
# the cached real LLM. Either way the call shape and downstream schema are
# identical.

final_subtypes = []
final_sources = []
final_confidences = []
final_versions = []

for i in range(len(df_corpus)):
    rule_st = list(df_corpus["subtype_rule"].iloc[i])
    kw_st = list(df_corpus["subtype_keyword"].iloc[i])
    kw_new = list(df_corpus["subtype_keyword_new"].iloc[i])

    if len(rule_st) > 0 and len(kw_new) > 0:
        final_subtypes.append(sorted(set(rule_st) | set(kw_new)))
        final_sources.append("rule+keyword")
        final_confidences.append(1.0)
        final_versions.append(None)
    elif len(rule_st) > 0:
        final_subtypes.append(sorted(set(rule_st)))
        final_sources.append("rule")
        final_confidences.append(1.0)
        final_versions.append(None)
    elif len(kw_st) > 0:
        final_subtypes.append(sorted(set(kw_st)))
        final_sources.append("keyword")
        final_confidences.append(1.0)
        final_versions.append(None)
    else:
        sts, conf, ver = subtype_classifier(df_corpus["utterance"].iloc[i])
        final_subtypes.append(sts)
        final_sources.append("llm")
        final_confidences.append(conf)
        final_versions.append(ver)

df_corpus["subtype"] = final_subtypes
df_corpus["subtype_source"] = final_sources
df_corpus["subtype_confidence"] = final_confidences
df_corpus["subtype_prompt_version"] = final_versions
df_corpus = df_corpus.drop(columns=["subtype_rule", "subtype_keyword", "subtype_keyword_new"])

src_counts = Counter(final_sources)
print("Subtype source distribution:")
for src in ("rule", "rule+keyword", "keyword", "llm"):
    n = src_counts.get(src, 0)
    print(f"  {src:14s}: {n:3d} ({n/len(df_corpus):.1%})")

n_llm = src_counts.get("llm", 0)
if n_llm > 0:
    if USE_REAL_LLM_STEP2:
        print(f"\n  {n_llm} rows classified by real Llama 3.3-70b "
              f"(prompt_version=subtype_v1_llama70b, cached).")
    else:
        n_stubbed = (df_corpus["subtype_prompt_version"] == "subtype_v0_stub").sum()
        print(f"\n  {n_stubbed} rows assigned via STUB "
              f"(default 'observation', confidence=0.0). "
              f"Set USE_REAL_LLM_STEP2=True above to replace with real classifications.")
else:
    print(f"\n  No rows hit the LLM fallback (every utterance had >=1 cue).")

In [ ]:
assert df_corpus["subtype"].apply(lambda s: len(s) > 0).all(), \
    "DoD violation: every utterance must have >=1 sub-type"
print("Every row has >=1 sub-type.")

flat = [st for sts in df_corpus["subtype"] for st in sts]
subtype_counts = Counter(flat)
n_rows = len(df_corpus)

print(f"\nSub-type distribution (multi-label, {n_rows} rows total):")
warnings = []
for st in SUBTYPES_ALL:
    n = subtype_counts.get(st, 0)
    pct = n / n_rows
    flag = ""
    if n == 0:
        flag = "  <-- MISSING (0 examples)"
        warnings.append(st)
    elif pct > 0.70:
        flag = f"  <-- OVER-DOMINANT ({pct:.1%} > 70%)"
        warnings.append(st)
    print(f"  {st:18s}: {n:3d} rows ({pct:.1%}){flag}")

PRACTICE_LABELS = {"observation", "prediction", "causal_reasoning", "evidence"}
has_practice = df_corpus["subtype"].apply(
    lambda s: any(st in PRACTICE_LABELS for st in s)
)
content_only = df_corpus["subtype"].apply(
    lambda s: set(s) == {"content"}
)
print(f"\nPractice-label coverage:")
print(f"  Rows with >=1 practice label:    {has_practice.sum()} ({has_practice.mean():.1%})")
print(f"  Rows with ONLY 'content':        {content_only.sum()} ({content_only.mean():.1%})")
print(f"  (low practice coverage limits stratification options in Step 7)")

print(f"\nMulti-label statistics:")
sizes = df_corpus["subtype"].apply(len)
print(f"  rows with 1 subtype:  {(sizes == 1).sum()}")
print(f"  rows with 2 subtypes: {(sizes == 2).sum()}")
print(f"  rows with 3+ subtypes: {(sizes >= 3).sum()}")
print(f"  mean subtypes/row: {sizes.mean():.2f}")

if warnings:
    print(f"\nDistribution warnings on: {warnings}")
    print(f"  Inspect manually before moving to Step 3. These are not hard failures")
    print(f"  ('content' is expected to be high since most positives carry a Tier3 noun);")
    print(f"  but a 0-count practice class means the seed taxonomy or LLM stub needs revisiting.")
else:
    print(f"\nDistribution looks sane (no class missing, none over 70%).")

In [ ]:
df_corpus.to_parquet(PROCESSED_DIR / "corpus.parquet", index=False)
print(f"Saved corpus.parquet with subtype columns.")

reread = pd.read_parquet(PROCESSED_DIR / "corpus.parquet")
new_cols = [c for c in reread.columns if c.startswith("subtype")]
print(f"New columns: {new_cols}")
print(f"Total schema: {list(reread.columns)}")

for src in ("rule", "rule+keyword", "keyword", "llm"):
    rows = reread[reread["subtype_source"] == src]
    if len(rows) == 0:
        continue
    print(f"\nSample '{src}' assignments ({len(rows)} total, showing 3):")
    for _, r in rows.head(3).iterrows():
        snippet = r["utterance"][:75] + ("..." if len(r["utterance"]) > 75 else "")
        extra = f" conf={r['subtype_confidence']:.2f}" if src == "llm" else ""
        print(f"  {r['utt_id']}: {list(r['subtype'])}{extra}")
        print(f"    '{snippet}'")

In [ ]:
def test_subtype_schema():
    """Step 2 schema test: validates the new subtype columns on disk."""
    df = pd.read_parquet(PROCESSED_DIR / "corpus.parquet")

    required = {"subtype", "subtype_source", "subtype_confidence", "subtype_prompt_version"}
    missing = required - set(df.columns)
    assert not missing, f"Missing subtype columns: {missing}"

    assert df["subtype"].apply(lambda s: len(s) > 0).all(), \
        "Every utterance must have >=1 sub-type (DoD violation)"

    valid = set(SUBTYPES_ALL)
    seen = {st for sts in df["subtype"] for st in sts}
    assert seen.issubset(valid), f"Unexpected subtype values: {seen - valid}"

    valid_sources = {"rule", "rule+keyword", "keyword", "llm"}
    bad_sources = set(df["subtype_source"].unique()) - valid_sources
    assert not bad_sources, f"Unexpected subtype_source values: {bad_sources}"

    assert df["subtype_confidence"].between(0.0, 1.0).all(), \
        "subtype_confidence must be in [0, 1]"

    llm_rows = df[df["subtype_source"] == "llm"]
    if len(llm_rows) > 0:
        assert llm_rows["subtype_prompt_version"].notna().all(), \
            "LLM-assigned rows must carry a non-null subtype_prompt_version"

    non_llm = df[df["subtype_source"] != "llm"]
    if len(non_llm) > 0:
        assert (non_llm["subtype_confidence"] == 1.0).all(), \
            "rule/keyword rows should have confidence=1.0"

    print("Step 2 schema test PASSED")
    print(f"  rows:               {len(df)}")
    print(f"  subtype values:     {sorted(seen)}")
    print(f"  source distribution: {dict(Counter(df['subtype_source']))}")
    print(f"  confidence: min={df['subtype_confidence'].min():.2f} "
          f"mean={df['subtype_confidence'].mean():.3f}")

test_subtype_schema()

# Step 3 — Negative mining (gating step)

Build a `NOT_SCIENCE_TALK` pool from three source types and write `data/processed/negatives.parquet`.

**DoD:**
1. At least three negative source types are representable:
   - `transcript_clean` — transcript utterances with no seed-word match (requires transcripts)
   - `llm_hard_negative` — LLM-generated, anchored to a positive utterance
   - `seed_word_nonscience` — LLM-generated, anchored to a seed term used non-scientifically
2. Final pool ≥ 3× positives, balanced across sub-types
3. 50-row hand-check sample → ≥90% true-negative rate

The notebook drives `src/negatives_3.py`. Use the `stub_llm_callable` for fast iteration; switch to `make_real_llm_callable()` (Llama 3.3-70b) when you're ready to consume API quota.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/home/vashist/Desktop/Desktop/Anita Zucker Center/Science talk retrieval")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.negatives_3 import (
    build_negative_pool,
    compute_review_pass_rate,
    sample_for_review,
    stub_llm_callable,
    validate_negatives,
)

df_corpus = pd.read_parquet(PROCESSED_DIR / "corpus.parquet")
df_seed = pd.read_parquet(PROCESSED_DIR / "seed_words.parquet")
df_cat = pd.read_parquet(PROCESSED_DIR / "category_defs.parquet")

print(f"Loaded corpus={len(df_corpus)} positives={(df_corpus['label']=='SCIENCE_TALK').sum()}")
print(f"Loaded seed terms={len(df_seed)}, category defs={len(df_cat)}")

In [ ]:
# Build the negative pool.
#
# transcript_xlsx: hand-coded workbook. Per-sheet logic mines teacher utterances
#   (col C ~ 'T<n>:'), drops rows where col G contains 'SCI', and tags survivors
#   as either 'behavior_disapproving' (col E == 'Y') or 'other_teacher_talk'.
# max_transcript_negatives caps the transcript pool so it doesn't drown out the
#   LLM-generated sources (set to None to keep all ~7k candidates).
#
# LLM mode: stub by default (free, deterministic). For real Llama 3.3-70b output:
#     from src.negatives_3 import make_real_llm_callable
#     llm = make_real_llm_callable(model="llama-3.3-70b-instruct")
#     ... llm_callable=llm

from src.negatives_3 import DEFAULT_TRANSCRIPT_XLSX

df_neg = build_negative_pool(
    df_corpus, df_seed, df_cat,
    transcript_xlsx=DEFAULT_TRANSCRIPT_XLSX if DEFAULT_TRANSCRIPT_XLSX.exists() else None,
    max_transcript_negatives=1500,
    n_hard_per_positive=2,
    n_per_seed_term=2,
    seed_term_sample=None,
    llm_callable=stub_llm_callable,
    verbose=True,
)
df_neg.head(8)

In [ ]:
# DoD validation: pool size relative to positives, source-type coverage,
# sub-type balance. Returns warnings (does not raise) so you can iterate.
n_pos = int((df_corpus["label"] == "SCIENCE_TALK").sum())
info = validate_negatives(df_neg, n_positives=n_pos, target_ratio=3.0, verbose=True)

In [ ]:
# 50-row hand-check sample, stratified across source_type.
# Workflow:
#   1. Inspect each row's `text` column.
#   2. Set human_verified=True and human_label='NOT_SCIENCE_TALK' if it's a true
#      negative, else human_label='SCIENCE_TALK'.
#   3. Merge marks back into df_neg by `neg_id`.
#   4. Call compute_review_pass_rate(df_neg) and confirm pass_rate >= 0.90.

review = sample_for_review(df_neg, n=50, seed=42)
print(f"Pulled {len(review)} rows for hand-check; per source_type:")
print(review["source_type"].value_counts().to_dict())
review[["neg_id", "source_type", "anchor_seed_term", "text"]].head(10)

In [ ]:
out_path = PROCESSED_DIR / "negatives.parquet"
df_neg.to_parquet(out_path, index=False)
print(f"Wrote {out_path} ({out_path.stat().st_size:,} bytes, {len(df_neg)} rows)")

In [ ]:
def test_negatives_schema():
    df = pd.read_parquet(PROCESSED_DIR / "negatives.parquet")
    expected = {
        "neg_id", "text", "source_type", "subtype",
        "anchor_utt_id", "anchor_seed_term",
        "structural_check_passed", "llm_gate_score",
        "model_id", "prompt_version",
        "human_verified", "human_label",
    }
    missing = expected - set(df.columns)
    assert not missing, f"missing columns: {missing}"

    assert df["neg_id"].is_unique, "neg_id must be unique"
    assert df["text"].notna().all(), "text must be non-null"
    assert all(len(s) >= 1 for s in df["subtype"]), "every row needs >=1 subtype"

    valid_sources = {"transcript_clean", "llm_hard_negative", "seed_word_nonscience"}
    assert set(df["source_type"]).issubset(valid_sources), f"unknown source_type: {set(df['source_type']) - valid_sources}"

    print(f"Schema OK. {len(df)} rows, source distribution: {df['source_type'].value_counts().to_dict()}")

test_negatives_schema()

# Step 4 — LLM register-variant augmentation

Multiply each positive anchor with paraphrases in different teacher registers (large group, small group, informal/centers). The bi-encoder (Step 8) then learns invariance to register rather than overfitting to surface form.

**DoD:**
1. Each anchor has variants in ≥2 registers different from its source (anchors with `setting=Unknown` fan out to all 3).
2. Tier2/Tier3 cues preserved in ≥80% of variants.
3. No variant is identical to its anchor.
4. Every row records `model_id` and `prompt_version`.

The notebook drives `src/augment_4.py`. Default LLM mode is the deterministic stub. Switch to `make_real_llm_callable()` (Llama 3.3-70b) when you want real paraphrases.

In [ ]:
from src.augment_4 import (
    build_variant_pairs,
    sample_pairs_for_review,
    stub_llm_callable as stub_llm_callable_v4,
    validate_variant_pairs,
)

df_corpus_v4 = pd.read_parquet(PROCESSED_DIR / "corpus.parquet")
print(f"Loaded corpus={len(df_corpus_v4)}, "
      f"positives={(df_corpus_v4['label']=='SCIENCE_TALK').sum()}")

In [ ]:
# Build the variant pool.
#
# n_per_register controls how many variants the LLM is asked for per target
# register. With n=1 you get ~516 variants from the 191 positives (134 Unknown
# anchors fan out to all 3 registers; 57 known-setting anchors fan out to 2).
#
# Real LLM run: swap the stub for the real Llama callable:
#     from src.augment_4 import make_real_llm_callable
#     llm = make_real_llm_callable(model="llama-3.3-70b-instruct")
#     ... llm_callable=llm

df_pairs = build_variant_pairs(
    df_corpus_v4,
    n_per_register=1,
    llm_callable=stub_llm_callable_v4,
    verbose=True,
)
df_pairs.head(8)

In [ ]:
# DoD validation.
# - DoD #1 enforced as an assertion (each anchor has variants in >=2 other registers).
# - DoD #2 reported as a warning if cue preservation drops below 80%.
# - DoD #3 enforced as an assertion (no surface-form leak).
# - DoD #4 enforced as an assertion (model_id + prompt_version present).
info_v4 = validate_variant_pairs(df_pairs, df_corpus_v4, verbose=True)

In [ ]:
# 50-row hand-check sample, stratified across registers.
review_v4 = sample_pairs_for_review(df_pairs, n=50, seed=42)
print(f"Pulled {len(review_v4)} variants for hand-check; per register:")
print(review_v4["register"].value_counts().to_dict())
review_v4[["pair_id", "register", "anchor_text", "variant_text",
           "preservation_pct", "llm_self_score"]].head(10)

In [ ]:
out_path_v4 = PROCESSED_DIR / "pairs.parquet"
df_pairs.to_parquet(out_path_v4, index=False)
print(f"Wrote {out_path_v4} ({out_path_v4.stat().st_size:,} bytes, {len(df_pairs)} rows)")

In [ ]:
def test_pairs_schema():
    df = pd.read_parquet(PROCESSED_DIR / "pairs.parquet")
    expected = {
        "pair_id", "anchor_id", "anchor_text", "anchor_setting", "source_register",
        "variant_text", "register", "subtype",
        "preserved_cues", "preservation_pct", "preservation_check_passed",
        "differs_from_anchor",
        "llm_self_score", "prompt_version", "model_id", "created_at",
    }
    missing = expected - set(df.columns)
    assert not missing, f"missing columns: {missing}"

    assert df["pair_id"].is_unique, "pair_id must be unique"
    assert df["variant_text"].notna().all(), "variant_text must be non-null"
    assert df["differs_from_anchor"].all(), "no surface-form leak allowed (DoD #3)"
    assert df["model_id"].notna().all(), "model_id required (DoD #4)"
    assert df["prompt_version"].notna().all(), "prompt_version required (DoD #4)"

    valid_regs = {"LARGE_GROUP", "SMALL_GROUP", "INFORMAL"}
    assert set(df["register"]).issubset(valid_regs), \
        f"unknown registers: {set(df['register']) - valid_regs}"

    print(f"Schema OK. {len(df)} rows, register distribution: "
          f"{df['register'].value_counts().to_dict()}")

test_pairs_schema()

# Inspect generated artifacts

Quick eyeball of every parquet under `data/processed/` to sanity-check the pipeline output. Run after Steps 1–4.

In [ ]:
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

ARTIFACTS = {
    "corpus":       PROCESSED_DIR / "corpus.parquet",
    "seed_words":   PROCESSED_DIR / "seed_words.parquet",
    "category_defs":PROCESSED_DIR / "category_defs.parquet",
    "negatives":    PROCESSED_DIR / "negatives.parquet",
    "pairs":        PROCESSED_DIR / "pairs.parquet",
}

dfs = {}
for name, path in ARTIFACTS.items():
    if not path.exists():
        print(f"  [missing] {name}: {path.name} (run earlier steps first)")
        continue
    dfs[name] = pd.read_parquet(path)
    print(f"  {name:13} {dfs[name].shape[0]:>5} rows × {dfs[name].shape[1]:>2} cols  ({path.stat().st_size:,} bytes)")

In [ ]:
corpus = dfs["corpus"]
print("columns:", list(corpus.columns))
print("\nlabel distribution:")
print(corpus["label"].value_counts(dropna=False).to_string())
print("\nsetting distribution:")
print(corpus["setting"].value_counts(dropna=False).to_string())
print("\nsubtype source breakdown:")
print(corpus["subtype_source"].value_counts(dropna=False).to_string())
print("\ncue coverage:")
print(f"  rows with tier2_cues: {(corpus['tier2_cues'].apply(len) > 0).sum()}")
print(f"  rows with tier3_cues: {(corpus['tier3_cues'].apply(len) > 0).sum()}")
corpus.head(5)

In [ ]:
seeds = dfs["seed_words"]
print("columns:", list(seeds.columns))
print("\nseeds per tier:")
print(seeds["tier"].value_counts().to_string())
print("\nseeds per category (top 10):")
print(seeds["category"].value_counts().head(10).to_string())
print("\nterms with the most variants:")
seeds_with_var = seeds[seeds["variants"].apply(len) > 0].copy()
seeds_with_var["n_variants"] = seeds_with_var["variants"].apply(len)
seeds_with_var.nlargest(8, "n_variants")[["term", "tier", "category", "n_variants", "variants"]]

In [ ]:
cats = dfs["category_defs"]
print("columns:", list(cats.columns))
cats

In [ ]:
if "negatives" in dfs:
    neg = dfs["negatives"]
    print("columns:", list(neg.columns))
    print("\nsource_type distribution:")
    print(neg["source_type"].value_counts().to_string())
    print("\ntranscript_subtype (transcript_clean rows only):")
    tx = neg[neg["source_type"] == "transcript_clean"]
    print(tx["transcript_subtype"].value_counts(dropna=False).to_string())
    print("\nnegatives per subtype (flattened):")
    print(neg["subtype"].apply(list).explode().value_counts().to_string())
    print("\ntop transcript files by # negatives mined:")
    print(tx["transcript_ref"].dropna().str.extract(r"^([^!]+)")[0]
          .value_counts().head(10).to_string())
    print("\nhuman review state:")
    print(f"  marked verified : {(neg['human_verified'] == True).sum()} / {len(neg)}")

In [ ]:
if "negatives" in dfs:
    parts = [g.sample(min(3, len(g)), random_state=0)
             for _, g in dfs["negatives"].groupby("source_type")]
    samples = pd.concat(parts, ignore_index=True)
    samples[["neg_id", "source_type", "transcript_subtype",
             "anchor_seed_term", "text"]]

In [ ]:
if "pairs" in dfs:
    pairs = dfs["pairs"]
    print("columns:", list(pairs.columns))
    print("\nregister distribution:")
    print(pairs["register"].value_counts().to_string())
    print("\nsetting -> register matrix (anchors fanning out):")
    print(pd.crosstab(pairs["anchor_setting"].fillna("Unknown"), pairs["register"]))
    print(f"\nunique anchors covered: {pairs['anchor_id'].nunique()} of "
          f"{(dfs['corpus']['label']=='SCIENCE_TALK').sum()} positives")
    print(f"surface-leak count: {(~pairs['differs_from_anchor']).sum()}")
    print(f"cue preservation rate: {pairs['preservation_check_passed'].mean():.1%}")
    print(f"mean LLM self-score:   {pairs['llm_self_score'].mean():.3f}")

In [ ]:
if "pairs" in dfs:
    sample_anchor = dfs["pairs"]["anchor_id"].iloc[0]
    print(f"All variants of anchor {sample_anchor!r}:")
    dfs["pairs"][dfs["pairs"]["anchor_id"] == sample_anchor][
        ["pair_id", "register", "anchor_text", "variant_text",
         "preservation_pct", "llm_self_score"]
    ]